In [0]:
from pyspark.sql import functions as F
from delta import DeltaTable

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','customers','Data Source')

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog,data_source)

In [0]:
df = spark.read.format('csv').option('inferSchema','true').option('header','true').load('/Volumes/fmcg/bronze/source_fmcg/customers/').withColumn('read_timestamp',F.current_timestamp()).select('*','_metadata.file_name','_metadata.file_size')
df.display()

In [0]:
df.printSchema()

In [0]:
df_bronze = df.write.format('delta').mode('overwrite').saveAsTable('fmcg.bronze.customers')

# **Silver Transformation**

In [0]:
df_bronze = spark.sql('select * from fmcg.bronze.customers')

In [0]:
df_duplicates = df_bronze.groupBy('customer_id').count().where('count > 1')

In [0]:
df_duplicates.display()

In [0]:
print('Rows before duplicates dropped',df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped',df_silver.count())

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col('customer_name')))
)

In [0]:
df_silver = df_silver.withColumn('customer_name',F.trim(F.col('customer_name')))

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu' : 'Bengaluru',
    'Bengalore' : 'Bengaluru',


    'Hyderabad' : 'Hyderabad',
    'Hyderbad' : 'Hyderabad',

    'NewDelhi' : 'New Delhi',
    'NewDheli' : 'New Delhi',
    'NewDelhee' : 'New Delhi' 
}

allowed = ['Bengaluru','Hyderabad','New Delhi']

df_silver = (
    df_silver.replace(city_mapping,subset=['city'])
    .withColumn('city',
                F.when(F.col('city').isNull(),None)
                 .when(F.col('city').isin(allowed),F.col('city'))
                 .otherwise(None)
                )
)

In [0]:
df_silver = df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(),'None')
    .otherwise(F.initcap('customer_name'))
)

In [0]:
df_silver.filter(F.col('city').isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate=False)

In [0]:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k,v) for k, v in customer_city_fix.items()],['customer_id','fixed_city']
) 

In [0]:
df_silver = (
    df_silver
    .join(df_fix,'customer_id','left')
    .withColumn(
        'city',
        F.coalesce('city','fixed_city')
    )
    .drop('fixed_city')
)

In [0]:
df_silver.display()

In [0]:
df_silver.withColumn('customer_id',F.col('customer_id').cast('string'))
print(df_silver.printSchema())

In [0]:
df_silver = (
    df_silver.withColumn(
        "customer",F.concat_ws("_",'customer_name',F.coalesce(F.col('city'),F.lit("Unknown")))
    )
    .withColumn('market',F.lit("India"))
    .withColumn('platform',F.lit("SportsBar"))
    .withColumn('Channel',F.lit('Acquisition'))
)


In [0]:
df_silver.write.format('delta')\
    .option("delta.enableChangeDataFeed",'true')\
        .option('mergeSchema','true')\
            .mode('overwrite')\
                .saveAsTable('fmcg.silver.customers')

Gold Layer Processing


In [0]:
df_silver = spark.sql('Select * from fmcg.silver.customers')
#df_silver.display()
df_gold = df_silver.select('customer_id','customer_name','city','customer','market','platform','Channel')

In [0]:
df_gold.write\
    .format('delta')\
        .option('delta.enableChangeDataFeed','true')\
                .mode('overwrite')\
                    .saveAsTable('fmcg.gold.sports_bar_dim_customers')

In [0]:
delta_table = DeltaTable.forName(spark, 'fmcg.gold.dim_customers')
df_child_customers = spark.table('fmcg.gold.sports_bar_dim_customers').select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'Channel'
)

In [0]:
delta_table.alias('target').merge(
    source = df_child_customers.alias('source'),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()